In [ ]:
import asyncio
import html
import json
import os
import platform
import struct
import threading
import traceback

import ipywidgets
from IPython.display import display

import xyg
from geo_five_temporal_fixture import five_hierarchy_widgets
from xyg._geo_hierarchy import is_hierarchy_frame
from xyg._geo_temporal import _BINDINGS

source, scope, lanes, widgets, chart = five_hierarchy_widgets()
controller = xyg.TemporalController(instance_id=81, domain=(-(1 << 63), (1 << 63) - 1))
owner_loops = []
loop_threads = []
if os.environ.get("XYG_GEO_MULTILOOP") == "1":
    for _ in range(2):
        loop = asyncio.new_event_loop()
        thread = threading.Thread(target=loop.run_forever, daemon=True)
        owner_loops.append(loop)
        loop_threads.append(thread)
        thread.start()


def make_bindings():
    if not owner_loops:
        return [widget.bind_temporal(controller) for widget in widgets]

    async def bind(widget):
        return widget.bind_temporal(controller)

    return [
        asyncio.run_coroutine_threadsafe(bind(widget), owner_loops[i % 2]).result()
        for i, widget in enumerate(widgets)
    ]


async def in_owner(binding, method):
    if binding._loop is asyncio.get_running_loop():
        return await getattr(binding, method)()
    return await asyncio.wrap_future(
        asyncio.run_coroutine_threadsafe(getattr(binding, method)(), binding._loop)
    )


bindings = make_bindings()
outputs = [ipywidgets.Output() for _ in widgets]
status = ipywidgets.HTML()
display(status, *outputs)
for output, widget in zip(outputs, widgets, strict=True):
    with output:
        display(widget)
controller.set_cursor(3)
phases = []


def launch(action):
    global phase_task
    phase_task = asyncio.create_task(action())

    def completed(task):
        if task.cancelled():
            status.value = "ERROR cancelled"
        elif task.exception():
            status.value = "ERROR " + html.escape(
                "".join(traceback.format_exception(task.exception()))
            )

    phase_task.add_done_callback(completed)


async def idle():
    await asyncio.gather(*(in_owner(binding, "when_idle") for binding in bindings))
    assert all(binding.error is None for binding in bindings), [
        binding.error for binding in bindings
    ]


async def check(instant):
    await idle()
    assert len({widget._geo_mount for widget in widgets}) == 5
    assert all(widget._geo_mount for widget in widgets)
    for widget in widgets:
        adapter = widget._adapter
        assert adapter._source is source
        assert adapter._frame.data.identity["time"]["instant"] == instant
        assert adapter._frame.data.selection["ids"].tolist() == [(1 << 64) - 1]
        assert adapter._frame.data.visible_vertices == 2
        assert is_hierarchy_frame(adapter._frame)
        assert not adapter._live_candidate._owns_slot


async def ready():
    await check(3)
    status.value = "XYG five ready"


launch(ready)
camera_step = 0
camera_plan = [index for index in range(5) for _ in range(index + 1)]
last_camera = [0 for _ in widgets]


async def camera_next():
    global camera_step
    index = camera_plan[camera_step]
    widget = widgets[index]
    while widget._adapter._frame.data.identity["camera"]["center_x"] == last_camera[index] or any(
        w._adapter._live_candidate._owns_slot for w in widgets
    ):
        await asyncio.sleep(0)
    last_camera[index] = widget._adapter._frame.data.identity["camera"]["center_x"]
    camera_step += 1
    status.value = "XYG five camera " + str(camera_step)

In [ ]:
async def supersede():
    global held, send, active_index
    await idle()
    event = asyncio.Event()
    main_loop = asyncio.get_running_loop()
    held = []
    active_index = 0
    widget = widgets[active_index]
    send = widget.send

    def hold(message, buffers=()):
        if (
            not held
            and message.get("type") == "geo_host"
            and buffers
            and bytes(buffers[0][:4]) == b"XYGH"
            and struct.unpack_from("<I", buffers[0], 8)[0] == 6
        ):
            held.append((message, buffers))
            main_loop.call_soon_threadsafe(event.set)
        else:
            send(message, buffers=buffers)

    widget.send = hold
    controller.set_cursor(4)
    await event.wait()
    widget.send = send
    for instant in range(10, 1010):
        controller.set_cursor(instant)
    assert sum(binding._active is not None for binding in bindings) == 1
    assert sum(widget._adapter._live_candidate._owns_slot for widget in widgets) == 1
    status.value = "XYG five prepared held"


launch(supersede)

In [ ]:
async def release_held():
    widget = widgets[active_index]
    widget.send = send
    message, buffers = held.pop()
    send(message, buffers=buffers)
    await check(1009)
    phases.append("actual prepared supersession ACK then five latest")
    status.value = "XYG five latest"


launch(release_held)

In [ ]:
launch(camera_next)

In [ ]:
async def clear_views():
    global cameras, old_mounts
    await idle()
    cameras = [dict(widget._adapter._frame.data.identity["camera"]) for widget in widgets]
    assert len({camera["center_x"] for camera in cameras}) == 5
    old_mounts = [widget._geo_mount for widget in widgets]
    await asyncio.gather(*(in_owner(binding, "aclose") for binding in bindings))
    for output in outputs:
        output.clear_output(wait=False)
    while any(widget._adapter.mounted or widget._geo_mount is not None for widget in widgets):
        await asyncio.sleep(0)
    assert all(widget._geo_mount is None for widget in widgets)
    status.value = "XYG five released"


launch(clear_views)

In [ ]:
controller.set_cursor(23)
bindings = make_bindings()
for output, widget in zip(outputs, widgets, strict=True):
    with output:
        display(widget)


async def remounted():
    await check(23)
    assert all(widget._geo_mount != old for widget, old in zip(widgets, old_mounts, strict=True))
    assert [widget._adapter._frame.data.identity["camera"] for widget in widgets] == cameras
    phases.append("five actual output clear release ACK and redisplay ready")
    status.value = "XYG five remounted"


launch(remounted)

In [ ]:
async def finish():
    global report
    await idle()
    await asyncio.gather(*(in_owner(binding, "aclose") for binding in bindings))
    for widget in widgets:
        widget.close()
    while any(widget._adapter.mounted or widget._geo_mount is not None for widget in widgets):
        await asyncio.sleep(0)
    for widget in widgets:

        async def drain(widget):
            if widget._geo_rpc_task:
                await widget._geo_rpc_task

        if widget._geo_loop is asyncio.get_running_loop():
            await drain(widget)
        else:
            await asyncio.wrap_future(
                asyncio.run_coroutine_threadsafe(drain(widget), widget._geo_loop)
            )
        assert widget._adapter._anchor is None
    assert all(lane not in _BINDINGS for lane in lanes)
    source.close()
    for lane in lanes:
        lane.close()
    scope.close()
    controller.close()
    report = {
        "python": platform.python_version(),
        "package": xyg.__file__,
        "nativePath": os.environ["XYG_NATIVE_LIB"],
        "phases": phases,
        "views": 5,
        "canonicalSources": 1,
        "selectedId": str((1 << 64) - 1),
        "actualReleaseACKs": 5,
        "cameras": cameras,
        "owningLoops": len({id(binding._loop) for binding in bindings}),
    }
    for loop in owner_loops:
        loop.call_soon_threadsafe(loop.stop)
    for thread in loop_threads:
        thread.join()
    for loop in owner_loops:
        loop.close()
    status.value = "XYG five native cleanup acknowledged"


launch(finish)

In [ ]:
print("XYG five report " + json.dumps(report))